# 5. Case study: steam methane reforming vs. alkaline water electrolysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/05_case_study_hydrogen.ipynb)

Time to put everything together. Two plants each produce **50 t of hydrogen per day** on the US Gulf Coast: a **steam methane reforming (SMR)** plant sized from an Aspen Plus simulation, and a pressurized **alkaline water electrolysis (WE)** plant costed from a US DOE report. You will build both, compare their cost structures, construct a scenario in which electrolysis becomes competitive, and compare the two under uncertainty.

The data are adapted from case study 1 of the OpenPyTEA repository, where a third
pathway, methane pyrolysis, is also analysed. The SMR plant is given: run through it quickly. The electrolysis plant and the comparison are your tasks.

## Learning goals

- Build two plants from a flowsheet, sizing data, and vendor quotes.
- Compare their cost structures, cash flows, and sensitivities.
- Construct a realistic scenario in which electrolysis is cheaper than SMR.
- Estimate the probability that one technology is cheaper than the other.

## 0) Setup

In [ ]:
# Run this cell. It installs OpenPyTEA if it is missing (e.g., on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
import numpy as np
import pandas as pd
from copy import deepcopy
from openpytea import Equipment, Plant
from openpytea.analysis import *
from openpytea.plotting import *

## 1) Common assumptions

Both plants share the location, financing, and price assumptions below. Prices already include uncertainties
for the Monte Carlo comparison at the end.

In [ ]:
country, region   = "United States", "Gulf Coast"
interest_rate     = 0.09
project_lifetime  = 30          # years
plant_utilization = 0.95
tax_rate          = 0.25

h2_production = 50_000          # kg/day (50 t/day)
h2_price      = 5.0             # USD/kg

operator_rate = {"rate": 38.11,  # USD/h
                 "rate_uncertainty": {"std": 10, "min": 25, "max": 50}}

# Prices with their uncertainty: (base, std, min, max)
electricity_price = {
    "price": 0.05,  # USD/kWh
    "price_uncertainty": {"std": 0.03, "min": 0.01, "max": 3.0}}
natural_gas_price = {
    "price": 0.40,  # USD/kg
    "price_uncertainty": {"std": 0.25, "min": 0.0, "max": 5.0}}
cooling_water_price = {
    "price": 2.4592e-4,  # USD/MJ
    "price_uncertainty": {"std": 1e-4, "min": 1e-5, "max": 4e-4}}
CO_2_tax = {
    "price": 0.075,  # USD/kg CO2
    "price_uncertainty": {"std": 0.02, "min": 0.0, "max": 0.15}}
water_price = {
    "price": 0.003,  # USD/kg
    "price_uncertainty": {"std": 0.001, "min": 0.0, "max": 0.005}}

## 2) Steam methane reforming

Methane is reformed with steam at 700 °C over a nickel catalyst, the syngas is shifted in high- and
low-temperature water-gas-shift reactors, and hydrogen is recovered by pressure-swing adsorption. The tail gas
and extra methane fire the reformer furnace.

![SMR flowsheet](assets/smr_pfd.png)

In [ ]:
# Compressors and their electric motors (90 % motor efficiency)
comp_1 = Equipment("COMP-1", 945, "Fluids", "Compressors, fans, & blowers",
                   "Compressor, centrifugal", "Carbon steel")
comp_2 = Equipment("COMP-2", 491, "Fluids", "Compressors, fans, & blowers",
                   "Compressor, centrifugal", "Carbon steel")
mtr_1  = Equipment("MTR-1", 945 / 0.9, "Electrical", "Motors & generators",
                   "Totally enclosed motor", "Carbon steel")
mtr_2  = Equipment("MTR-2", 491 / 0.9, "Electrical", "Motors & generators",
                   "Totally enclosed motor", "Carbon steel")

# Heat exchangers (m2)
hx_1   = Equipment("HX-1",   31.87,  "Fluids", "Heat exchangers",
                   "U-tube shell & tube", "304 stainless steel")
hx_2   = Equipment("HX-2",   22.74,  "Fluids", "Heat exchangers",
                   "U-tube shell & tube", "304 stainless steel")
hx_3   = Equipment("HX-3",   107.19, "Fluids", "Heat exchangers",
                   "U-tube shell & tube", "304 stainless steel")
cooler = Equipment("COOLER", 191.47, "Fluids", "Heat exchangers",
                   "U-tube shell & tube", "Carbon steel")

# Reactors (shell mass, kg) and furnace (duty, kW), pump (L/s)
smr_reactor    = Equipment("SMR-REACTOR",    160,     "Fluids", "Reactors",
                           "Tubular fixed bed", "304 stainless steel")
wgs_ht_reactor = Equipment("WGS-HT-REACTOR", 376.54,  "Fluids", "Reactors",
                           "Tubular fixed bed", "304 stainless steel")
wgs_lt_reactor = Equipment("WGS-LT-REACTOR", 3822.92, "Fluids", "Reactors",
                           "Tubular fixed bed", "304 stainless steel")
furnace = Equipment("FURNACE", 33.26, "Fluids", "Boilers, heaters, & furnaces",
                    "Furnace, cylindrical", "304 stainless steel")
pump    = Equipment("PUMP",    5.16,  "Fluids", "Pumps",
                    "Single-stage centrifugal", "Carbon steel")

# PSA: no verified correlation in the database, so a literature cost (1994
# USD) is used
psa = Equipment("PSA", 1023.49, "Fluids", "Adsorption & membrane separation",
                "Pressure-swing adsorber", "Carbon steel",
                purchased_cost=284_692.37, cost_year=1994)

smr_equipment = [comp_1, mtr_1, comp_2, mtr_2, hx_1, hx_2, hx_3, cooler,
                 smr_reactor, wgs_ht_reactor, wgs_lt_reactor,
                 furnace, pump, psa]

In [ ]:
smr_plant = Plant({
    "plant_name": "Steam reforming",
    "process_type": "Fluids",
    "country": country, "region": region,
    "equipment": smr_equipment,
    "interest_rate": interest_rate, "project_lifetime": project_lifetime,
    "plant_utilization": plant_utilization, "tax_rate": tax_rate,
    "operator_hourly_rate": operator_rate,
    "plant_products": {
        "hydrogen": {"production": h2_production, "price": h2_price},
    },
    # daily quantities from the simulation: kWh, kg methane (feed + fuel),
    # MJ, kg CO2 emitted
    "variable_opex_inputs": {
        "electricity":   {"consumption": (945 + 491) / 0.9 * 24,
                          **electricity_price},
        "methane_feed":  {"consumption": (5139 + 4025) * 24,
                          **natural_gas_price},
        "cooling_water": {"consumption": 20_222.67 * 24,
                          **cooling_water_price},
        "CO_{2}_tax":   {"consumption": 25_124.64 * 24,
                         **CO_2_tax},
    },
})
smr_plant.calculate_all()
print(f"SMR  FCI {smr_plant.fixed_capital/1e6:6.1f} MUSD   "
      f"LCOH {smr_plant.levelized_cost:.2f} USD/kg   "
      f"NPV {smr_plant.npv/1e6:7.1f} MUSD   IRR {smr_plant.irr:.1%}")

## 3) Task 5.1: build the water-electrolysis plant

The DOE report describes a pressurized alkaline system (about 30 bar) delivering 99.99 % hydrogen without downstream compression.

![WE flowsheet](assets/we_pfd.png)

Instead of sized units, the report gives **system costs** (2020 USD): electrolyzer stack 17.194 MUSD, mechanical balance of plant (BOP) 11.019 MUSD, electrical BOP 16.974 MUSD. The system consumes **98 MW of electricity** and **7.56 kg of water per kg H₂** (process plus cooling make-up).

**Tasks**

1. Create three `Equipment` items from the quotes (`purchased_cost=..., cost_year=...`, `param=0`). Use the correct `process_type` for each system.
2. Create `we_plant` with the common assumptions, `process_type="Fluids"`, hydrogen as product, and electricity and water as variable OPEX inputs (reuse `electricity_price` and `water_price`).
3. The automatic operator estimate counts equipment items, which is meaningless for three "systems". The flowsheet has 11 fluid process steps: compute `we_plant.calculate_operators_hired(no_fluid_process=11, no_solid_process=0)`
   and set it with `update_configuration({"operators_hired": ...})`.
4. The stack lasts 10 years, so replace it in years 10 and 20: set `additional_capex_cost` to the stack's purchased cost twice and `additional_capex_years` to `[10, 20]`.
5. Run `we_plant.calculate_all(additional_capex=True)` and print FCI, LCOH, NPV, and IRR.

In [ ]:
# YOUR CODE HERE

## 4) Comparing cost structures

Pass both plants as a list. For the electrolysis plant, `additional_capex=True` includes the stack replacements
in the capital breakdown.

In [ ]:
plants = [smr_plant, we_plant]
fig, ax = plot_stacked_bar(fixed_capital_data(plants, additional_capex=True))

In [ ]:
fig, ax = plot_stacked_bar(variable_opex_data(plants))

In [ ]:
fig, ax = plot_stacked_bar(levelized_cost_data(plants))

In [ ]:
fig, ax = plot_cash_flow(cash_flow_data(plants))

> Electrolysis is dominated by capital and electricity; reforming by the methane feed and, to a lesser extent, the CO₂ tax. At today's assumptions SMR produces hydrogen at about 3.7 USD/kg and electrolysis at about 6.8 USD/kg.

## 5) Task 5.2: what would change the ranking?

At today's assumptions SMR is clearly cheaper. Use sensitivity analysis to find out what it would take for electrolysis to compete. Pass `additional_capex=True` to `sensitivity_data` so the stack replacements are counted.

**Tasks**

1. Play around with `sensitivity_data` and `plot_sensitivity` for both plants, e.g. `"electricity"`, `"methane_feed"`, or `"fixed_capital"`. Which inputs move the gap the most? Can a single realistic change close it?
2. Construct a **realistic scenario** in which electrolysis is cheaper than SMR: `deepcopy` the plants, change several inputs with `update_configuration` (think of power prices, gas prices, CO₂ taxes, or government support), and show that the electrolysis LCOH ends up below the SMR one.

In [ ]:
# YOUR CODE HERE

## 6) Task 5.3: how sure are we about the ranking?

All prices already have uncertainties, and the default project-level ones are active.

1. Run `monte_carlo` on both plants with 20 000 samples and the same `random_seed` (pass `additional_capex=True` for electrolysis).
2. Plot both LCOH distributions on one figure with `plot_multiple_monte_carlo`.
3. Compute the probability that electrolysis is cheaper than SMR: `np.mean(we < smr)`.
4. Bonus: what is the probability if the electricity price is a long-term contract at 0.02 USD/kWh with `std` 0.005?

In [ ]:
# YOUR CODE HERE

## 7) Discussion

We pick these questions up together in the wrap-up. If you finish early, discuss them with your neighbour:

- Which inputs would you spend effort on before showing these results to a decision maker?
- The comparison uses a **equipment-level estimate** for SMR and **system-level costs** for WE. Is that a fair comparison? How would you make the accuracy classes consistent?
- What does your scenario tell a policy maker, and what does it hide (e.g., upstream methane emissions, electricity carbon intensity, or learning-curve of electrolyzers)?

## Recap

- You built two plants from process data and compared their costs and drivers.
- You constructed a scenario in which electrolysis wins and compared the plants under uncertainty.

**Next:** [6. JSON configuration files and the command line](06_json_and_cli.ipynb)

After the workshop:

- Try the [home assignment](../HOME_ASSIGNMENT.md) on a process of your own.
- Missing a cost correlation? Contribute it: https://github.com/pbtamarona/OpenPyTEA